# Run the global Rossby model

This notebook is a local validation run, not a test. It opens the supplied 1000 m isobath and ERA5–SCOTIA forcing products, coarsens only the spatial forcing grid for a quick calculation, and runs the complete temporal `GlobalRossbyModel.solve()` path. Monthly samples are treated as uniformly spaced by `365.25 / 12` days while their original calendar labels are retained on output.

In [ ]:
from pathlib import Path
from os import PathLike, fspath
from dask.diagnostics import ProgressBar
import sys

import numpy as np
import xarray as xr

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from moc_adjustment_theory import GlobalRossbyModel

ISOBATH_PATH = ROOT / "data/tracked/isobath/global_isobath_GEBCO_1000m.nc"
FORCING_PATH = ROOT / "data/untracked/forcing/global_ERA5_SCOTIA_forcing_island_inpainted.nc"
MONTH_SECONDS = 365.25 / 12 * 24 * 60 * 60
G_PRIME = 0.02

def save_dask_dataset_to_zarr(
    dataset: xr.Dataset,
    path: str | PathLike[str],
    *,
    overwrite: bool = False,
) -> None:
    """Stream a Dask-backed xarray Dataset to a Zarr store."""
    write = dataset.to_zarr(
        fspath(path),
        mode="w" if overwrite else "w-",
        zarr_format=2,
        consolidated=True,
        compute=False,
    )

    with ProgressBar():
        write.compute()

In [ ]:
isobath = xr.open_dataset(ISOBATH_PATH)
forcing = xr.open_dataset(FORCING_PATH)

In [ ]:
model = GlobalRossbyModel(isobath, g_prime=G_PRIME)
solution = model.solve(
    forcing,
    sample_spacing_seconds = MONTH_SECONDS
)

In [ ]:
solution.attrs['isobath'] = str(ISOBATH_PATH)
solution.attrs['forcing'] = str(FORCING_PATH)

In [ ]:
solution

In [ ]:
save_dask_dataset_to_zarr(solution, ROOT / "data/untracked/model_output/global_rossby_model_solution.zarr", overwrite=True)